<a href="https://colab.research.google.com/github/mszhdgr/Machine_learning/blob/Main/Pyspark/pyspark_knn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ahmedmohamed2003/cafe-sales-dirty-data-for-cleaning-training")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'cafe-sales-dirty-data-for-cleaning-training' dataset.
Path to dataset files: /kaggle/input/cafe-sales-dirty-data-for-cleaning-training


In [69]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Caffe Sales").getOrCreate()
spark

In [70]:
df = spark.read.csv("/kaggle/input/cafe-sales-dirty-data-for-cleaning-training/dirty_cafe_sales.csv", inferSchema=True, header=True)

In [71]:
df.show()

+--------------+--------+--------+--------------+-----------+--------------+--------+----------------+
|Transaction ID|    Item|Quantity|Price Per Unit|Total Spent|Payment Method|Location|Transaction Date|
+--------------+--------+--------+--------------+-----------+--------------+--------+----------------+
|   TXN_1961373|  Coffee|       2|           2.0|        4.0|   Credit Card|Takeaway|      2023-09-08|
|   TXN_4977031|    Cake|       4|           3.0|       12.0|          Cash|In-store|      2023-05-16|
|   TXN_4271903|  Cookie|       4|           1.0|      ERROR|   Credit Card|In-store|      2023-07-19|
|   TXN_7034554|   Salad|       2|           5.0|       10.0|       UNKNOWN| UNKNOWN|      2023-04-27|
|   TXN_3160411|  Coffee|       2|           2.0|        4.0|Digital Wallet|In-store|      2023-06-11|
|   TXN_2602893|Smoothie|       5|           4.0|       20.0|   Credit Card|    NULL|      2023-03-31|
|   TXN_4433211| UNKNOWN|       3|           3.0|        9.0|         ERR

In [72]:
df.take(5)

[Row(Transaction ID='TXN_1961373', Item='Coffee', Quantity='2', Price Per Unit='2.0', Total Spent='4.0', Payment Method='Credit Card', Location='Takeaway', Transaction Date='2023-09-08'),
 Row(Transaction ID='TXN_4977031', Item='Cake', Quantity='4', Price Per Unit='3.0', Total Spent='12.0', Payment Method='Cash', Location='In-store', Transaction Date='2023-05-16'),
 Row(Transaction ID='TXN_4271903', Item='Cookie', Quantity='4', Price Per Unit='1.0', Total Spent='ERROR', Payment Method='Credit Card', Location='In-store', Transaction Date='2023-07-19'),
 Row(Transaction ID='TXN_7034554', Item='Salad', Quantity='2', Price Per Unit='5.0', Total Spent='10.0', Payment Method='UNKNOWN', Location='UNKNOWN', Transaction Date='2023-04-27'),
 Row(Transaction ID='TXN_3160411', Item='Coffee', Quantity='2', Price Per Unit='2.0', Total Spent='4.0', Payment Method='Digital Wallet', Location='In-store', Transaction Date='2023-06-11')]

In [73]:
df.show(5)

+--------------+------+--------+--------------+-----------+--------------+--------+----------------+
|Transaction ID|  Item|Quantity|Price Per Unit|Total Spent|Payment Method|Location|Transaction Date|
+--------------+------+--------+--------------+-----------+--------------+--------+----------------+
|   TXN_1961373|Coffee|       2|           2.0|        4.0|   Credit Card|Takeaway|      2023-09-08|
|   TXN_4977031|  Cake|       4|           3.0|       12.0|          Cash|In-store|      2023-05-16|
|   TXN_4271903|Cookie|       4|           1.0|      ERROR|   Credit Card|In-store|      2023-07-19|
|   TXN_7034554| Salad|       2|           5.0|       10.0|       UNKNOWN| UNKNOWN|      2023-04-27|
|   TXN_3160411|Coffee|       2|           2.0|        4.0|Digital Wallet|In-store|      2023-06-11|
+--------------+------+--------+--------------+-----------+--------------+--------+----------------+
only showing top 5 rows


In [74]:
from pyspark.sql import functions as F

df.sort(F.col('Quantity')).show(5)

+--------------+--------+--------+--------------+-----------+--------------+--------+----------------+
|Transaction ID|    Item|Quantity|Price Per Unit|Total Spent|Payment Method|Location|Transaction Date|
+--------------+--------+--------+--------------+-----------+--------------+--------+----------------+
|   TXN_2265316|  Cookie|    NULL|           1.0|        3.0|   Credit Card|In-store|      2023-12-29|
|   TXN_7609853|Sandwich|    NULL|           4.0|        4.0|Digital Wallet|In-store|      2023-04-22|
|   TXN_6319728|  Coffee|    NULL|           2.0|        4.0|   Credit Card|In-store|      2023-07-18|
|   TXN_7533411|  Cookie|    NULL|           1.0|        1.0|Digital Wallet|In-store|      2023-11-09|
|   TXN_4152036|   ERROR|    NULL|           3.0|        6.0|          Cash|    NULL|         UNKNOWN|
+--------------+--------+--------+--------------+-----------+--------------+--------+----------------+
only showing top 5 rows


In [75]:
df.printSchema()

root
 |-- Transaction ID: string (nullable = true)
 |-- Item: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Price Per Unit: string (nullable = true)
 |-- Total Spent: string (nullable = true)
 |-- Payment Method: string (nullable = true)
 |-- Location: string (nullable = true)
 |-- Transaction Date: string (nullable = true)



In [76]:
df.describe().show()

+-------+--------------+-------+-----------------+------------------+-----------------+--------------+--------+----------------+
|summary|Transaction ID|   Item|         Quantity|    Price Per Unit|      Total Spent|Payment Method|Location|Transaction Date|
+-------+--------------+-------+-----------------+------------------+-----------------+--------------+--------+----------------+
|  count|         10000|   9667|             9862|              9821|             9827|          7421|    6735|            9841|
|   mean|          NULL|   NULL|3.028463396702027| 2.949984155487483|8.924352495262161|          NULL|    NULL|            NULL|
| stddev|          NULL|   NULL|1.419006873221319|1.2784504728035881|6.009919472829945|          NULL|    NULL|            NULL|
|    min|   TXN_1000555|   Cake|                1|               1.0|              1.0|          Cash|   ERROR|      2023-01-01|
|    max|   TXN_9999124|UNKNOWN|          UNKNOWN|           UNKNOWN|          UNKNOWN|       UNK

In [77]:
df.select(["Item","Location"]).summary().show()

+-------+-------+--------+
|summary|   Item|Location|
+-------+-------+--------+
|  count|   9667|    6735|
|   mean|   NULL|    NULL|
| stddev|   NULL|    NULL|
|    min|   Cake|   ERROR|
|    25%|   NULL|    NULL|
|    50%|   NULL|    NULL|
|    75%|   NULL|    NULL|
|    max|UNKNOWN| UNKNOWN|
+-------+-------+--------+



In [78]:
df.select(F.mode(F.column("Item"))).show()

+----------+
|mode(Item)|
+----------+
|     Juice|
+----------+



In [79]:
df.select(F.mode(F.column("Item"))).first()[0]

'Juice'

In [80]:
df = df.na.fill(df.select(F.mode(F.column("Item"))).first()[0], subset="Item")

In [81]:
df.select(F.mode(F.column("Location"))).show()

+--------------+
|mode(Location)|
+--------------+
|      Takeaway|
+--------------+



In [82]:
df = df.na.fill(df.select(F.mode(F.column("Location"))).first()[0], subset="Location")

In [83]:
#df = df.na.fill("Missing").na.fill(0)

In [84]:
df = df.na.drop(how="any")

In [85]:
df.summary().show()

+-------+--------------+-------+------------------+------------------+------------------+--------------+--------+----------------+
|summary|Transaction ID|   Item|          Quantity|    Price Per Unit|       Total Spent|Payment Method|Location|Transaction Date|
+-------+--------------+-------+------------------+------------------+------------------+--------------+--------+----------------+
|  count|          6969|   6969|              6969|              6969|              6969|          6969|    6969|            6969|
|   mean|          NULL|   NULL|3.0270149918361287|2.9668405365126675|  8.98239488931808|          NULL|    NULL|            NULL|
| stddev|          NULL|   NULL|1.4195092193832937|1.2789134522436294|6.0217493841511835|          NULL|    NULL|            NULL|
|    min|   TXN_1000555|   Cake|                 1|               1.0|               1.0|          Cash|   ERROR|      2023-01-01|
|    25%|          NULL|   NULL|               2.0|               2.0|             

In [90]:
df.select(F.col("Transaction Date")).where(F.col("Transaction Date") == "ERROR").count()

98

In [94]:
df = df.where(F.col("Transaction Date") != "ERROR")

df.summary().show()

+-------+--------------+-------+------------------+------------------+-----------------+--------------+--------+----------------+
|summary|Transaction ID|   Item|          Quantity|    Price Per Unit|      Total Spent|Payment Method|Location|Transaction Date|
+-------+--------------+-------+------------------+------------------+-----------------+--------------+--------+----------------+
|  count|          6871|   6871|              6871|              6871|             6871|          6871|    6871|            6871|
|   mean|          NULL|   NULL| 3.025440313111546|2.9665860296341093|8.973093156466687|          NULL|    NULL|            NULL|
| stddev|          NULL|   NULL|1.4199879016858112| 1.279667769841769|6.021688614900909|          NULL|    NULL|            NULL|
|    min|   TXN_1000555|   Cake|                 1|               1.0|              1.0|          Cash|   ERROR|      2023-01-01|
|    25%|          NULL|   NULL|               2.0|               2.0|              4.0|  

In [100]:
df.select(F.col("Transaction Date")).where(F.col("Transaction Date") == "ERROR").count()

0

In [104]:
df = df.where(F.col("Transaction Date") != "ERROR")

for column in df.columns:
  df = df.where(F.col(column) != "NULL")
for column in df.columns:
  df = df.where(F.col(column) != "UNKNOWN")
for column in df.columns:
  df = df.where(F.col(column) != "MISSING")
for column in df.columns:
  df = df.where(F.col(column) != "ERROR")

In [105]:
df.select(F.col("Payment Method")).where(F.col("Payment Method") == "NULL").count()

0

In [106]:
df.summary().show()

+-------+--------------+----+------------------+------------------+-----------------+--------------+--------+----------------+
|summary|Transaction ID|Item|          Quantity|    Price Per Unit|      Total Spent|Payment Method|Location|Transaction Date|
+-------+--------------+----+------------------+------------------+-----------------+--------------+--------+----------------+
|  count|          4877|4877|              4877|              4877|             4877|          4877|    4877|            4877|
|   mean|          NULL|NULL|3.0287061718269426|2.9729341808488825|9.019889276194382|          NULL|    NULL|            NULL|
| stddev|          NULL|NULL|1.4185560614280706|1.2792057587544436|6.007507791479432|          NULL|    NULL|            NULL|
|    min|   TXN_1000555|Cake|                 1|               1.0|              1.0|          Cash|In-store|      2023-01-01|
|    25%|          NULL|NULL|               2.0|               2.0|              4.0|          NULL|    NULL|  

In [107]:
df.show(10)

+--------------+--------+--------+--------------+-----------+--------------+--------+----------------+
|Transaction ID|    Item|Quantity|Price Per Unit|Total Spent|Payment Method|Location|Transaction Date|
+--------------+--------+--------+--------------+-----------+--------------+--------+----------------+
|   TXN_1961373|  Coffee|       2|           2.0|        4.0|   Credit Card|Takeaway|      2023-09-08|
|   TXN_4977031|    Cake|       4|           3.0|       12.0|          Cash|In-store|      2023-05-16|
|   TXN_3160411|  Coffee|       2|           2.0|        4.0|Digital Wallet|In-store|      2023-06-11|
|   TXN_2602893|Smoothie|       5|           4.0|       20.0|   Credit Card|Takeaway|      2023-03-31|
|   TXN_2548360|   Salad|       5|           5.0|       25.0|          Cash|Takeaway|      2023-11-07|
|   TXN_7619095|Sandwich|       2|           4.0|        8.0|          Cash|In-store|      2023-05-03|
|   TXN_2847255|   Salad|       3|           5.0|       15.0|   Credit Ca

In [109]:
df.filter(df.Item == "Coffee").show()

+--------------+------+--------+--------------+-----------+--------------+--------+----------------+
|Transaction ID|  Item|Quantity|Price Per Unit|Total Spent|Payment Method|Location|Transaction Date|
+--------------+------+--------+--------------+-----------+--------------+--------+----------------+
|   TXN_1961373|Coffee|       2|           2.0|        4.0|   Credit Card|Takeaway|      2023-09-08|
|   TXN_3160411|Coffee|       2|           2.0|        4.0|Digital Wallet|In-store|      2023-06-11|
|   TXN_6420335|Coffee|       1|           2.0|        2.0|          Cash|Takeaway|      2023-07-16|
|   TXN_3748616|Coffee|       2|           2.0|        4.0|   Credit Card|In-store|      2023-12-09|
|   TXN_3011323|Coffee|       3|           2.0|        6.0|   Credit Card|Takeaway|      2023-03-07|
|   TXN_3085509|Coffee|       4|           2.0|        8.0|Digital Wallet|In-store|      2023-04-15|
|   TXN_8779771|Coffee|       4|           2.0|        8.0|          Cash|In-store|      20

In [121]:
df.groupBy("Location").sum().count()

2